# Trace a LangChain research agent

You will build a [LangChain](https://www.langchain.com/) agent with two tools — Tavily web search,
and a local calculator it is told to use for any arithmetic — and run it twice in one conversation:
once to find a price, once to split that price between four people. Every chain, tool and model
call lands in AcruxCore automatically.

**There is no AcruxCore code in the agent.** Not one line. That makes this notebook the odd one out
in this set: no prompts to version, no tool catalog, no gateway. LangChain runs its own agent loop,
its own tool dispatch and its own model calls, and reports what it did over
[OpenTelemetry](https://opentelemetry.io/) — the protocol nearly every agent framework speaks.
AcruxCore accepts that protocol at `POST /api/v1/traces/otlp`, so pointing a working agent at us is
three lines of setup outside the agent.

| Piece | What it does | Who runs it |
|---|---|---|
| `register()` | builds a standard OTel pipeline aimed at our OTLP endpoint | **your code**, once at startup |
| `openinference-instrumentation-langchain` | records every chain, tool and model call | a library, patching LangChain |
| `create_agent(...)` | runs the agent loop and decides which tool to call | **LangChain**, knowing nothing about us |
| `run_name` | gives the trace a findable name instead of `LangGraph` | **your code** |
| `using_session(...)` | ties both turns together as one conversation | **your code** |

Every cell runs against a real account, the real OpenAI API and the real Tavily API. Nothing here
is faked or mocked.

**One instrumentor, not two.** If you have read the [CrewAI notebook](../../trace-a-crewai-trip-planner/notebook/trip_planner.ipynb),
this is the difference worth carrying over: that one needs `["crewai", "openai"]`, this one needs
`["langchain"]` alone. Step 1 explains why, and Step 9 shows what adding the second one does to
your trace.

**Two ways to do the one step that changes something.** Step 2 has two headings: **In the
dashboard**, with the value to set, and **The same thing in code**, with a cell to run. They are not
two different features — the dashboard and the SDK call the same API, so the result is identical.
Pick either; doing both is harmless, because the code cell reads the current value before it writes.

**Three kinds of code cell.** Most of this notebook is not the thing you would ship. Every cell's
lead-in says which kind it is:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | creates or changes something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or shows what just happened | no |
| **Broken on purpose** | a failure being demonstrated | no |

**Companion page:** [Trace a LangChain Research Agent](https://docs.acruxcore.com/docs/tutorials/trace-a-langchain-research-agent)

---

## Step 0 — What you need before you start

**1. A personal AcruxCore API key.** **Account & keys → New key**. Copy it the moment it appears —
that is the only time the full value is shown.

**2. An OpenAI API key.** The agent calls `gpt-4o-mini` directly, not through our gateway, so this
key goes to OpenAI and never to us.

Nothing here needs OpenAI in particular. Swap `ChatOpenAI` for `ChatAnthropic` or any other
LangChain chat model and the tracing below is unaffected, because the instrumentor watches
LangChain rather than the provider.

**3. A Tavily API key.** The agent searches the web with it. The free tier is enough.

**4. Python 3.10 or newer.** This notebook was run on 3.14. Unlike CrewAI, LangChain has no upper
version bound to worry about here.

**5. Five packages.** One is our OTel helper, one is the instrumentor that does the actual
recording, and three are LangChain itself plus the two integrations the agent uses.

`langchain` gives you `create_agent` and `tool`. `langchain-openai` gives you `ChatOpenAI`.
`langchain-tavily` gives you `TavilySearch` — and unlike CrewAI's Tavily wrapper it brings its own
HTTP client, so there is no second package to remember.

In [ ]:
%pip install -q --upgrade langchain langchain-openai langchain-tavily \
    "acruxcore[otel]" openinference-instrumentation-langchain

**Setup.** Set the three keys.

A key typed into a notebook is saved *inside the notebook file*. Prefer setting these in your shell
before you start Jupyter, and treat this cell as a fallback.

In [1]:
import os

os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")
os.environ.setdefault("OPENAI_API_KEY", "sk-...")
os.environ.setdefault("TAVILY_API_KEY", "tvly-...")

MODEL = "gpt-4o-mini"
SESSION_ID = "langchain-notebook-demo"

# Do NOT print the base URL: the saved output would publish whatever host you ran against.
print("env set")

env set


### Preflight

**Check.** Four things, in the order they fail. Read the instrumentor version even when it passes:
every span name in this notebook comes from that library and from LangChain, and both change
between releases.

In [2]:
import importlib.metadata as metadata

from acruxcore import AcruxCore

hub = AcruxCore()

await hub.traces.list(limit=1)
print("acruxcore api key: ok")

for package in ("langchain", "langchain-openai", "langchain-tavily",
                "openinference-instrumentation-langchain", "acruxcore"):
    print(f"  {package:42} {metadata.version(package)}")

for name, prefix in (("OPENAI_API_KEY", "sk-"), ("TAVILY_API_KEY", "tvly-")):
    value = os.environ[name]
    # Length matters as much as the prefix: the placeholder above starts with the right
    # prefix, so a prefix-only check passes and the run fails ten cells later instead.
    assert value.startswith(prefix) and len(value) > 20, f"{name} is still a placeholder"
print("openai + tavily keys: present")

acruxcore api key: ok
  langchain                                  1.4.0
  langchain-openai                           1.6.0
  langchain-tavily                           0.2.18
  openinference-instrumentation-langchain    0.1.74
  acruxcore                                  0.8.0
openai + tavily keys: present


---

## Step 1 — Who owns the agent loop

### The general problem

An observability tool can only describe work it knows about, and there are exactly two ways it can
find out.

It can **run the loop**. Your code asks the platform to call the model, so the platform sees every
request and response and writes the trace itself. That is what most tutorials on this site do,
through the gateway.

Or the framework can **run its own loop and report afterwards**. LangChain has its own agent graph,
its own tool dispatch and its own model wrappers. It is never going to hand that to somebody else's
SDK. What it does instead — like CrewAI, LlamaIndex and the OpenAI Agents SDK — is emit
OpenTelemetry spans describing what it did.

The second shape is not a lesser version of the first. It is the only shape available once the
framework owns the loop.

### Where our case sits

`POST /api/v1/traces/otlp` accepts OTLP directly. `acruxcore.otel.register()` is a small helper that
builds the pipeline every OTel app needs — a `TracerProvider`, a `BatchSpanProcessor` and an
`OTLPSpanExporter` — pointed at `$ACRUXCORE_BASE_URL/traces/otlp`, using `$ACRUXCORE_API_KEY` as the
bearer token. It does nothing you could not write by hand in four lines.

The instrumentor is a separate package from the
[OpenInference](https://github.com/Arize-ai/openinference) project, not ours. It patches LangChain
and emits spans carrying OpenInference attributes. Our OTLP endpoint reads those attributes and maps
them onto the same `chain` / `agent` / `tool` / `llm` span kinds every other tutorial produces —
which is why a LangChain trace looks at home next to a gateway trace.

### The direct answer: why one instrumentor is enough here

The CrewAI notebook needs two names, `["crewai", "openai"]`, and says so loudly. This one needs
`["langchain"]` alone. The reason is where the model call happens.

CrewAI orchestrates agents itself but reaches the model through the plain `openai` SDK, one layer
below anything `crewai` instrumentation can see. So the model call — and with it the model name,
the token counts and the cost — is invisible unless you also patch `openai`.

LangChain does not have that gap. Everything it does, including the `ChatOpenAI` call, is routed
through one internal object: `CallbackManager`. The LangChain instrumentor patches that single
object, so chain, tool and LLM spans all arrive from one name.

Adding `"openai"` on top of it is not harmless belt-and-braces. Both instrumentors then see the
same model call. The LangChain one reports it inside your agent's trace, where it belongs; the
`openai` one reports it again as a **separate trace of its own**, with no agent around it — and
carrying the same tokens and the same cost, so your usage totals count that call twice. Step 9 runs
it on purpose so you can see both traces.

### The trap

`register()` installs a process-wide `TracerProvider`, and OpenTelemetry allows exactly one of
those per process. Call it once. To change it in a notebook, restart the kernel — a second
`register()` hands back a provider that is not wired to your exporter.

---

## Step 2 — Turn on payload capture

This is the only thing this notebook changes on the platform, and it decides whether the most
interesting part of the trace exists at all: the search query the model chose, and what came back.

With capture off, you still get the shape — which tool ran, how long it took, whether it failed. You
do not get what went in or out. The setting is per team and applies to every trace, so it is a
deliberate choice about storing model and tool payloads, not a per-run flag.

### In the dashboard

**Observability → Settings.**

| Field | What to set |
|---|---|
| **Capture payloads** | on |

### The same thing in code

**Setup.** Read it first, and only write when it differs — an unnecessary write is still an audit
entry.

In [3]:
settings = await hub.traces.get_settings()
if settings.capture_payloads:
    print("capture_payloads already on — nothing to change")
else:
    settings = await hub.traces.update_settings(capture_payloads=True)
    print("capture_payloads turned on")

capture_payloads already on — nothing to change


---

## Step 3 — Register the OTel pipeline

**Your app.** Three lines, at the top of your entry point.

`service_name` is what OTel calls the emitting application. `instrument` names the layers to patch.
Keep the returned provider: it is the handle you flush with.

Run this cell **once**. If you re-run it, restart the kernel first — the trap at the end of Step 1.

In [4]:
from acruxcore.otel import register

tracer_provider = register(
    service_name="langchain-research-agent",
    instrument=["langchain"],          # one name covers chain, tool and LLM spans
)
print("otel pipeline registered")

otel pipeline registered


---

## Step 4 — Build the two tools

**Your app.** Plain LangChain. Nothing here knows AcruxCore exists.

`TavilySearch` is a real web search. `split_cost` is a local Python function — the interesting one,
because the system prompt in Step 5 forbids the model from doing arithmetic itself. That is what
makes the tool call happen reliably rather than occasionally, and it is what Step 10 breaks.

The docstring is not decoration: LangChain sends it to the model as the tool description, so the
model reads it when deciding whether this is the right tool.

In [5]:
from langchain_core.tools import tool
from langchain_tavily import TavilySearch


@tool
def split_cost(total_amount: float, people: int, months: int) -> str:
    """Split a total cost between people and across months.

    Args:
        total_amount: The full amount for ONE month, in any single currency.
        people: How many people share the cost.
        months: How many months the cost runs for.
    """
    if people <= 0 or months <= 0:
        return "people and months must both be greater than zero"
    grand_total = total_amount * months
    return (
        f"Total for {months} month(s): {grand_total:.2f}. "
        f"Per person per month: {total_amount / people:.2f}. "
        f"Per person for the whole {months} month(s): {grand_total / people:.2f}."
    )


search = TavilySearch(max_results=5)
print("tools ready:", [search.name, split_cost.name])

tools ready: ['tavily_search', 'split_cost']


---

## Step 5 — Build the agent

**Your app.** One call. `create_agent` builds the loop: call the model, run whatever tools it asked
for, call the model again with the results, repeat until it stops asking.

The agent is rebuilt per turn in the helper below so each run is independent. That is a choice about
this notebook, not a requirement — an agent object is reusable.

In [6]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

SYSTEM_PROMPT = (
    "You are a research assistant. Search the web for facts you do not already know, "
    "and name your sources. When a question involves splitting a cost between people "
    "or across months, you MUST use the split_cost tool rather than doing the "
    "arithmetic yourself."
)


def build_research_agent(tools):
    """Build the two-tool agent.

    A notebook helper, NOT an SDK function. It wraps the one real LangChain call,
    create_agent(), so the mistake cells in Step 9 can swap the tool list.
    """
    return create_agent(
        model=ChatOpenAI(model=MODEL, temperature=0),
        tools=tools,
        system_prompt=SYSTEM_PROMPT,
    )


print("agent builder ready")

agent builder ready


---

## Step 6 — Give the run a name

**Your app.** One config key, and the reason it matters is not obvious until you have a few runs.

LangChain builds the agent as a LangGraph graph, and names the root span after the graph class. So
without `run_name`, every trace you ever record is called `LangGraph` and your trace list becomes
unreadable. Step 9 shows exactly that.

`run_name` is a normal LangChain config key, not an AcruxCore one. It sets the root span's name,
which is what our trace list shows.

In [7]:
async def run_turn(messages, tools) -> str:
    """Invoke the agent once and return the final assistant text.

    A notebook helper, NOT an SDK function. The only line that matters for tracing is the
    run_name config, which names the root span.
    """
    result = await build_research_agent(tools).ainvoke(
        {"messages": messages},
        config={"run_name": "research-agent"},
    )
    return result["messages"][-1].content


print("run_turn ready")

run_turn ready


---

## Step 7 — Run both turns in one session

**Your app.** `using_session` puts a `session.id` attribute on every span produced inside the block.
AcruxCore groups traces sharing that id into one conversation. LangChain does not know this is
happening; the attribute rides along on the spans it was already emitting.

Turn 2 carries turn 1's real answer forward, so the follow-up is a genuine revision rather than a
second unrelated question. The model has the price it found and only has to divide it — which is
exactly the call the system prompt reserved for `split_cost`.

This cell makes real web searches and real model calls, so it takes a few seconds.

In [8]:
from openinference.instrumentation import using_session

TOOLS = [search, split_cost]

QUESTION_1 = (
    "What does a hot desk at Second Home Lisboa in Lisbon cost per month? "
    "Give the price and the source."
)
QUESTION_2 = (
    "Four of us want that hot desk for 3 months. What is the total, and the "
    "cost per person per month?"
)

with using_session(SESSION_ID):
    answer_1 = await run_turn([{"role": "user", "content": QUESTION_1}], TOOLS)
print("=== Turn 1 ===")
print(answer_1)

with using_session(SESSION_ID):
    answer_2 = await run_turn(
        [
            {"role": "user", "content": QUESTION_1},
            {"role": "assistant", "content": answer_1},
            {"role": "user", "content": QUESTION_2},
        ],
        TOOLS,
    )
print()
print("=== Turn 2 (follow-up) ===")
print(answer_2)

=== Turn 1 ===
A hot desk at Second Home Lisboa costs approximately €250 per month. 

Source: [Monis Rent](https://www.monis.rent/post/best-coworking-spaces-in-lisbon)

=== Turn 2 (follow-up) ===
The total cost for four people to use the hot desk for 3 months is €750. Each person will pay €62.50 per month, which totals €187.50 for the entire 3 months.


**Your app.** Flush before reading. `BatchSpanProcessor` holds spans and exports them in batches,
which is what makes OTel cheap enough to leave on always. The cost is that a read immediately after
a run can find nothing at all — Step 9 shows that happening.

In [9]:
tracer_provider.force_flush()
print("flushed")

flushed


---

## Step 8 — Read the traces back

**Check.** Not a screenshot — the same API the dashboard uses.

Two traces per run of this notebook, one per turn, both with real token counts and real cost.
Model output varies between runs, so **the numbers below will differ from yours**. The span shape
is the stable part.

In [10]:
session_traces = await hub.traces.list(session_id=SESSION_ID, limit=10)

# Newest first, so this run's two turns are always the first two - which keeps the cell
# correct when you re-run the notebook and the session grows by two more.
turn_2_summary, turn_1_summary = session_traces.data[0], session_traces.data[1]

print(f"traces in session {SESSION_ID}: {session_traces.total} (grows by 2 each run)")
for label, summary in (("turn 1", turn_1_summary), ("turn 2", turn_2_summary)):
    print(f"  {label}  {summary.name:16} spans={summary.span_count:3} "
          f"tokens={summary.total_tokens:6} cost=${summary.total_cost_usd}")

traces in session langchain-notebook-demo: 2 (grows by 2 each run)
  turn 1  research-agent   spans=  7 tokens=  4201 cost=$0.0006621
  turn 2  research-agent   spans=  7 tokens=  2941 cost=$0.0004713


That is the Sessions page, in three lines. In the dashboard the same thing looks like this:

![The langchain-research-agent-demo session page listing two research-agent traces with their span counts, token counts and cost](https://docs.acruxcore.com/img/tutorials/trace-a-langchain-research-agent/01-session.png)

**Check.** Now walk one trace's span tree. `hub.traces.get()` returns the root spans, each with its
own `.children`, so a recursive walk prints the shape LangChain actually executed.

In [11]:
def walk(spans, depth=0):
    """Print a span tree. A notebook helper, NOT an SDK function."""
    for span in spans:
        model = f"  model={span.model}" if span.model else ""
        tokens = f"  tokens={span.total_tokens}" if span.total_tokens else ""
        print(f"{'  ' * depth}[{span.kind:7}] {span.name}{model}{tokens}")
        walk(span.children, depth + 1)


turn_1 = await hub.traces.get(turn_1_summary.id)
print(f"{turn_1.trace.name}  status={turn_1.trace.status}  spans={turn_1.trace.span_count}")
walk(turn_1.spans)

research-agent  status=ok  spans=7
[agent  ] research-agent
  [chain  ] model
    [llm    ] ChatOpenAI  model=gpt-4o-mini-2024-07-18  tokens=1354
  [chain  ] tools
    [tool   ] tavily_search
  [chain  ] model
    [llm    ] ChatOpenAI  model=gpt-4o-mini-2024-07-18  tokens=2847


The `kind` column is the interesting part. LangChain emitted OpenInference attributes; our endpoint
mapped them onto the same span kinds a gateway trace uses. Nothing in the agent chose these.

The same tree in the dashboard:

![Trace detail for research-agent showing an agent root span, a chain span named model containing an LLM span for ChatOpenAI, a chain span named tools containing the tavily_search tool span, and a second model chain with its own ChatOpenAI LLM span](https://docs.acruxcore.com/img/tutorials/trace-a-langchain-research-agent/02-trace-tree.png)

**Check.** The payloads are the reason Step 2 exists. They arrive under `span.payload`, separate
from `span.attributes`, because the capture setting gates them — with capture off the dict is
empty and everything else on the span is unchanged. This prints the real search query the model
chose, straight out of the tool span.

In [12]:
def every_span(spans):
    """Flatten a span tree. A notebook helper, NOT an SDK function."""
    for span in spans:
        yield span
        yield from every_span(span.children)


for span in every_span(turn_1.spans):
    if span.kind == "tool":
        # Payloads live under .payload, not on the span itself, and the dict is empty when
        # the team setting from Step 2 is off. That separation is the capture gate.
        payload = span.payload or {}
        print(f"tool: {span.name}")
        print(f"  input:  {str(payload.get('input'))[:160]}")
        print(f"  output: {str(payload.get('output'))[:160]}")

tool: tavily_search
  input:  {"query": "hot desk price Second Home Lisboa Lisbon", "search_depth": "basic"}
  output: {"type": "tool", "data": {"content": "{\"query\": \"hot desk price Second Home Lisboa Lisbon\", \"follow_up_questions\": null, \"answer\": null, \"images\": [],


Clicking that span in the dashboard shows the same two fields:

![The expanded tavily_search span showing Input with a hot desk pricing query and Output with the raw JSON returned by Tavily](https://docs.acruxcore.com/img/tutorials/trace-a-langchain-research-agent/03-tool-span.png)

**Check.** Cost is computed by us, not reported by LangChain. The instrumentor sends the model name
and the token counts; our endpoint prices them.

In [13]:
for span in every_span(turn_1.spans):
    if span.kind == "llm":
        print(f"{span.name:12} model={span.model:26} provider={span.provider}")
        print(f"  prompt={span.prompt_tokens} completion={span.completion_tokens} "
              f"cost=${span.cost_usd}")

ChatOpenAI   model=gpt-4o-mini-2024-07-18     provider=openai
  prompt=1327 completion=27 cost=$0.00021525
ChatOpenAI   model=gpt-4o-mini-2024-07-18     provider=openai
  prompt=2803 completion=44 cost=$0.00044685


---

## Step 9 — Three ways to get this wrong

Every cell in this step is **broken on purpose**. None of it is app code. Each one runs a tiny
tool-free agent so it stays cheap.

### Mistake 1 — also instrumenting `openai`

**Broken on purpose.** The mistake you make by reading the CrewAI tutorial first. There, two
instrumentors are correct. Here, the second one sees the same model call the LangChain instrumentor
already saw, and reports it again.

This cell runs the same tiny agent twice — once correctly, once with the `openai` instrumentor also
on — and prints every trace each run produced. Exactly one model call happens each time.

If you have run this notebook before, both counts will be higher. What matters is that the second
is twice the first.

In [14]:
from openinference.instrumentation.openai import OpenAIInstrumentor


async def show_traces_produced(session_id: str) -> None:
    """Run one tiny agent and print every trace that run produced.

    A notebook helper, NOT an SDK function.
    """
    with using_session(session_id):
        await run_turn([{"role": "user", "content": "Say the word two."}], [])
    tracer_provider.force_flush()
    listed = await hub.traces.list(session_id=session_id, limit=10)
    print(f"  traces produced: {listed.total}")
    for summary in listed.data:
        print(f"    {summary.name:16} spans={summary.span_count}  "
              f"tokens={summary.total_tokens}  cost=${summary.total_cost_usd}")


print("one instrumentor (correct):")
await show_traces_produced("langchain-notebook-one-instrumentor")

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)   # broken on purpose
print("two instrumentors (the mistake):")
await show_traces_produced("langchain-notebook-two-instrumentors")

OpenAIInstrumentor().uninstrument()      # put it back before anything else runs
print("openai instrumentor removed again")

one instrumentor (correct):
  traces produced: 1
    research-agent   spans=3  tokens=67  cost=$1.095e-05
two instrumentors (the mistake):
  traces produced: 2
    research-agent   spans=3  tokens=67  cost=$1.095e-05
    ChatCompletion   spans=1  tokens=67  cost=$1.095e-05
openai instrumentor removed again


One model call, two traces. The extra one is called `ChatCompletion`, holds a single span, and
carries the same token count and the same cost as the real call inside the agent trace — so every
model call your agent makes is counted twice in your usage and cost totals, and the duplicate has
no agent, no tools and no session context around it to explain what it was.

Nothing errors, and the correct trace is still there and still correct. That is what makes this one
worth demonstrating rather than describing.

### Mistake 2 — forgetting `run_name`

**Broken on purpose.** Nothing errors. The trace arrives complete, with every span, token and
dollar intact. It is just called `LangGraph`, like every other trace you will ever record this way,
so you cannot find it again.

In [15]:
with using_session("langchain-notebook-no-run-name"):
    # broken on purpose: no config={"run_name": ...}
    await build_research_agent([]).ainvoke(
        {"messages": [{"role": "user", "content": "Say the word two."}]}
    )
tracer_provider.force_flush()

unnamed = await hub.traces.list(session_id="langchain-notebook-no-run-name", limit=1)
print(f"trace name without run_name: {unnamed.data[0].name!r}")
print(f"trace name with run_name:    {session_traces.data[0].name!r}")

trace name without run_name: 'LangGraph'
trace name with run_name:    'research-agent'


### Mistake 3 — reading the trace before flushing

**Broken on purpose.** `BatchSpanProcessor` is the reason OTel is cheap enough to leave on in
production, and this is the price. A read straight after a run can find nothing.

The same trap is far worse in a script than in a notebook: a process that exits without flushing
loses whatever was still queued, with no error.

In [16]:
with using_session("langchain-notebook-flush-timing"):
    await run_turn([{"role": "user", "content": "Say the word two."}], [])

before = await hub.traces.list(session_id="langchain-notebook-flush-timing", limit=10)
print(f"traces found BEFORE force_flush(): {before.total}")

tracer_provider.force_flush()

after = await hub.traces.list(session_id="langchain-notebook-flush-timing", limit=10)
print(f"traces found AFTER  force_flush(): {after.total}")

traces found BEFORE force_flush(): 0
traces found AFTER  force_flush(): 1


---

## Step 10 — What a failing tool actually costs you

**Broken on purpose**, and the most important cell in this notebook.

Everything so far showed tracing working when the agent works. Here the calculator breaks. Watch
what the *user* gets back, not just what the trace says.

In [17]:
@tool
def split_cost_broken(total_amount: float, people: int, months: int) -> str:
    """Split a total cost between people and across months.

    Args:
        total_amount: The full amount for ONE month, in any single currency.
        people: How many people share the cost.
        months: How many months the cost runs for.
    """
    raise RuntimeError("upstream pricing API returned 503")     # broken on purpose


try:
    broken_answer = await run_turn(
        [{"role": "user", "content":
          "A desk costs 250 EUR per month. Four of us want it for 3 months. "
          "What is the total, and the cost per person per month?"}],
        [split_cost_broken],
    )
    print("the agent answered anyway:")
    print(broken_answer)
except Exception as exc:
    print(f"the run stopped: {type(exc).__name__}: {str(exc).splitlines()[0]}")

the run stopped: RuntimeError: upstream pricing API returned 503


**Check.** Whatever happened above, the trace records it. Read the tool span's status.

In [18]:
tracer_provider.force_flush()

recent = await hub.traces.list(limit=5)
broken_trace = await hub.traces.get(recent.data[0].id)   # the run above is the newest
print(f"trace status: {broken_trace.trace.status}")
for span in every_span(broken_trace.spans):
    marker = "ERROR" if span.status == "error" else "ok   "
    lines = (span.error_message or "").splitlines()
    first_line = lines[0][:70] if lines else ""
    print(f"  {marker} [{span.kind:7}] {span.name:22} {first_line}")

trace status: error
  ERROR [agent  ] research-agent         RuntimeError('upstream pricing API returned 503')Traceback (most recen
  ok    [chain  ] model                  
  ok    [llm    ] ChatOpenAI             
  ERROR [chain  ] tools                  RuntimeError('upstream pricing API returned 503')Traceback (most recen
  ERROR [tool   ] split_cost_broken      RuntimeError('upstream pricing API returned 503')Traceback (most recen


The correct answers are 750 EUR total and 62.50 EUR per person per month. Check what the model
actually said above.

In Python the exception usually propagates and the run stops, which is loud and easy to notice. In
Node, `createAgent` catches the tool error and feeds it back to the model, which then does the
arithmetic itself — exactly what the system prompt forbade — and returns a confident wrong answer
with no error anywhere the user can see. That is what this trace, captured from the Node version of
the same agent, looks like: a green root span, two red `split_cost` spans, and a trace marked
**Error**.

![Trace detail titled research-agent with status Error, showing a green chain root span and two red split cost tool spans among otherwise green model request, ChatOpenAI and tools spans](https://docs.acruxcore.com/img/tutorials/trace-a-langchain-research-agent/04-error.png)

That is the whole argument for tracing an agent rather than logging its final answer. A run that
quietly degraded and a run that worked produce output that looks the same.

---

## Step 11 — Flush and close

**Your app.** The last thing your entry point does. In a script this goes at the end of `main()`;
here it is a cell.

`hub` is only used by this notebook's **Check** cells, but it holds an HTTP pool, so close it too.

In [19]:
tracer_provider.force_flush()
await hub.gateway.aclose()
print("flushed and closed")

flushed and closed


---

## What you built

A two-tool LangChain agent that searches the web and then does arithmetic with a tool, run twice as
one conversation, fully traced — chains, tools, model calls, tokens and cost — with the agent itself
knowing nothing about AcruxCore.

### What of this actually ships

Three lines above your agent, one config key, and one line at the end:

```python
from acruxcore.otel import register

tracer_provider = register(
    service_name="langchain-research-agent",
    instrument=["langchain"],                # one name, NOT ["langchain", "openai"]
)

from langchain.agents import create_agent
from openinference.instrumentation import using_session

# ... your agent, exactly as you already wrote it ...

def main() -> None:
    with using_session("langchain-research-agent-demo"):
        result = agent.invoke(
            {"messages": messages},
            config={"run_name": "research-agent"},   # or every trace is called LangGraph
        )

    tracer_provider.force_flush()                    # do not skip this
```

Note `ainvoke` in the cells above and `invoke` here. A script has no running event loop, so the
synchronous call is correct; a notebook, a web handler or anything already inside asyncio uses the
async one.

### What was scaffolding

Everything else. `build_research_agent`, `run_turn`, `walk`, `every_span` and `count_llm_spans` are
notebook helpers, not SDK functions — they exist so the cells can be re-run and re-read. The
`hub = AcruxCore()` client is only there so the **Check** cells can read traces back; your app does
not need it to send them. And every cell in Steps 9 and 10 is a failure being demonstrated.

### Where to go next

- The companion page: [Trace a LangChain Research Agent](https://docs.acruxcore.com/docs/tutorials/trace-a-langchain-research-agent)
- The same integration on a framework that needs two instrumentors: [Trace a CrewAI Trip-Planning Crew](https://docs.acruxcore.com/docs/tutorials/trace-a-crewai-trip-planner)
- The endpoint itself, including gzip, batching and error responses: [OTLP Trace Ingestion](https://docs.acruxcore.com/api-reference/traces/otlp)